# N-gram Language Model (Theory)

## 1. Problem statement
A language model assigns probabilities to sequences of words (tokens). For a word sequence
$$
w_1^n = w_1, w_2, \dots, w_n,
$$
the joint probability is
$$
P(w_1^n) = P(w_1)P(w_2 \mid w_1)P(w_3 \mid w_1^2)\cdots P(w_n \mid w_1^{n-1}).
$$

Direct estimation of these conditional probabilities is infeasible for long histories because of data sparsity. N-gram models approximate the condition using a fixed-size history.

## 2. Markov (N-gram) assumption
Under the \((N-1)\)-order Markov assumption (an **N-gram**),
$$
P(w_i \mid w_1^{i-1}) \approx P(w_i \mid w_{i-N+1}^{i-1}),
$$
so the probability of the whole sequence becomes
$$
P(w_1^n) \approx \prod_{i=1}^n P(w_i \mid w_{i-N+1}^{i-1}).
$$
(Shorter contexts are used at the sequence start, often by adding special tokens like \(\langle s\rangle\) and \(\langle /s\rangle\).)

## 3. Maximum likelihood estimation (MLE)
For an N-gram \(h = w_{i-N+1}^{i-1}\) and next word \(w\), the MLE estimate from corpus counts is
$$
\hat{P}_{\text{MLE}}(w \mid h) = \frac{C(h\,w)}{C(h)},
$$
where \(C(h\,w)\) is the count of the N-gram \(h\) followed by \(w\), and \(C(h)\) is the count of context \(h\).

## 4. Smoothing (Laplace / Add-one)
To avoid zero probabilities for unseen N-grams, add-\(\alpha\) smoothing is often used. For add-one (\(\alpha=1\)):
$$
\hat{P}_{\text{Laplace}}(w \mid h) = \frac{C(h\,w) + \alpha}{C(h) + \alpha V},
$$
where \(V\) is the vocabulary size.

More advanced techniques include Good-Turing, Kneser–Ney, and backoff/interpolation.

## 5. Interpolation / Backoff (brief)
Linear interpolation (for combining lower- and higher-order estimates) for a trigram model:
$$
\hat{P}(w_i\mid w_{i-2}w_{i-1}) = \lambda_3 \hat{P}_{\text{tri}} + \lambda_2 \hat{P}_{\text{bi}} + \lambda_1 \hat{P}_{\text{uni}},
$$
with \(\lambda_1+\lambda_2+\lambda_3=1\).

Backoff methods (e.g., Katz backoff) use higher-order estimates when available; otherwise they "back off" to lower-order estimates, with re-normalization.

## 6. Evaluation — Cross-entropy and Perplexity
Average ne


In [1]:
# N-gram language model (simple, educational)
# - build n-gram counts
# - Laplace (add-one) smoothing
# - log-probability and perplexity
# - simple sampling generation

from collections import defaultdict, Counter
import math
import random
import re

def tokenize(text):
    """Simple tokenizer: lowercase, split on whitespace and basic punctuation."""
    text = text.lower()
    # keep sentence boundary token markers if desired
    tokens = re.findall(r"\w+|[.!?]", text)
    return tokens

def add_sentence_tokens(sent_tokens, n):
    """Add (n-1) start tokens '<s>' and one end token '</s>'."""
    start = ['<s>'] * (n-1)
    return start + sent_tokens + ['</s>']

class NGramModel:
    def __init__(self, n, smoothing=1.0):
        assert n >= 1
        self.n = n
        self.smoothing = smoothing  # alpha for add-alpha smoothing
        self.ngram_counts = Counter()
        self.context_counts = Counter()
        self.vocab = set()
    
    def update(self, tokenized_sentences):
        """tokenized_sentences: list of lists of tokens (without added <s>, </s>)"""
        for sent in tokenized_sentences:
            toks = add_sentence_tokens(sent, self.n)
            self.vocab.update(toks)
            # generate n-grams
            for i in range(len(toks) - self.n + 1):
                ngram = tuple(toks[i:i+self.n])
                context = tuple(toks[i:i+self.n-1]) if self.n > 1 else ()
                self.ngram_counts[ngram] += 1
                self.context_counts[context] += 1
    
    def prob(self, word, context):
        """Return smoothed probability P(word | context).
           context: tuple of previous n-1 words. If fewer provided, use the last n-1 tokens.
        """
        if self.n == 1:
            context = ()
        else:
            # ensure context length = n-1
            context = tuple(context[-(self.n-1):]) if len(context) >= (self.n-1) else tuple(['<s>']*(self.n-1-len(context))+list(context))
        ngram = tuple(list(context) + [word])
        numer = self.ngram_counts.get(ngram, 0) + self.smoothing
        denom = self.context_counts.get(context, 0) + self.smoothing * len(self.vocab)
        return numer / denom
    
    def log_probability(self, tokens):
        """Return log2 probability of token sequence (including start/end tokens)."""
        toks = add_sentence_tokens(tokens, self.n)
        logp = 0.0
        for i in range(self.n - 1, len(toks)):
            context = tuple(toks[i-self.n+1:i]) if self.n > 1 else ()
            word = toks[i]
            p = self.prob(word, context)
            logp += math.log2(p)
        return logp
    
    def perplexity(self, sentences):
        """Compute perplexity over a list of token lists (no start/end tokens provided)."""
        total_log2 = 0.0
        total_words = 0
        for sent in sentences:
            toks = add_sentence_tokens(sent, self.n)
            # number of predicted words = len(toks) - (n-1)
            for i in range(self.n - 1, len(toks)):
                context = tuple(toks[i-self.n+1:i]) if self.n > 1 else ()
                word = toks[i]
                p = self.prob(word, context)
                total_log2 += math.log2(p)
                total_words += 1
        avg_neg_log2 = - total_log2 / total_words
        return 2 ** avg_neg_log2
    
    def generate(self, max_len=30, seed_context=None):
        """Generate one sentence/sample. seed_context: list or tuple of up to n-1 tokens."""
        if seed_context is None:
            context = ['<s>'] * (self.n - 1)
        else:
            context = list(seed_context)[- (self.n - 1) :]
            if len(context) < (self.n - 1):
                context = ['<s>'] * ((self.n - 1) - len(context)) + context
        output = []
        for _ in range(max_len):
            # build distribution over vocab
            candidates = list(self.vocab)
            probs = [self.prob(w, context) for w in candidates]
            # normalize (numerical safe)
            s = sum(probs)
            if s <= 0:
                break
            probs = [p/s for p in probs]
            # sample
            word = random.choices(candidates, weights=probs, k=1)[0]
            if word == '</s>':
                break
            output.append(word)
            if self.n > 1:
                context = context[1:] + [word]
        return output

# ---- Example usage ----

if __name__ == "__main__":
    sample_text = [
        "I love natural language processing.",
        "I love machine learning.",
        "Language models are fun to build.",
        "I build language models for fun."
    ]
    # tokenize sentences
    tokenized = [tokenize(s) for s in sample_text]
    # create trigram model (n=3) with add-one smoothing
    model = NGramModel(n=3, smoothing=1.0)
    model.update(tokenized)
    
    # show some probabilities
    context = ("i", "love")
    print("P('natural' | i love) =", model.prob("natural", context))
    print("P('machine'  | i love) =", model.prob("machine", context))
    
    # log-prob and perplexity on training data (toy example)
    perp = model.perplexity(tokenized)
    print("Perplexity on training data (toy):", perp)
    
    # generate text
    for i in range(5):
        gen = model.generate(max_len=15, seed_context=["i"])
        print("Generated:", " ".join(gen))


P('natural' | i love) = 0.1111111111111111
P('machine'  | i love) = 0.1111111111111111
Perplexity on training data (toy): 8.022134994546677
Generated: to build . natural learning models processing models . love processing . to <s>
Generated: to are natural <s>
Generated: . models fun language <s>
Generated: for processing
Generated: build build machine are . are for build fun to learning language <s> . natural


In [2]:
# #Unigram model
# Unigram: words are independent.

In [ ]:
from collections import Counter
import math
import random

class UnigramModel:
    def __init__(self):
        self.counts = Counter()
        self.total = 0
    
    def train(self, corpus):
        """corpus: list of tokens"""
        self.counts.update(corpus)
        self.total = sum(self.counts.values())
    
    def prob(self, word):
        """P(word)"""
        return self.counts[word] / self.total if self.total > 0 else 0
    
    def generate(self, n=10):
        """Sample n words independently"""
        words = list(self.counts.keys())
        freqs = [self.counts[w] for w in words]
        return random.choices(words, weights=freqs, k=n)

# --- Example ---
corpus = "I love NLP I love AI I study AI".lower().split()
uni = UnigramModel()
uni.train(corpus)

print("P('i') =", uni.prob("i"))
print("P('ai') =", uni.prob("ai"))
print("Generated:", " ".join(uni.generate(5)))


In [3]:
# Bigram: each word depends only on the previous one.

In [4]:
from collections import defaultdict, Counter
import random

class BigramModel:
    def __init__(self):
        self.bigram_counts = defaultdict(Counter)
        self.context_totals = Counter()
    
    def train(self, corpus):
        """corpus: list of tokens"""
        tokens = ["<s>"] + corpus + ["</s>"]  # add start/end
        for w1, w2 in zip(tokens[:-1], tokens[1:]):
            self.bigram_counts[w1][w2] += 1
            self.context_totals[w1] += 1
    
    def prob(self, w2, w1):
        """P(w2 | w1)"""
        return self.bigram_counts[w1][w2] / self.context_totals[w1] if self.context_totals[w1] > 0 else 0
    
    def generate(self, max_len=10):
        word = "<s>"
        result = []
        for _ in range(max_len):
            next_words = list(self.bigram_counts[word].keys())
            freqs = list(self.bigram_counts[word].values())
            if not next_words:
                break
            word = random.choices(next_words, weights=freqs, k=1)[0]
            if word == "</s>":
                break
            result.append(word)
        return result

# --- Example ---
corpus = "I love NLP I love AI I study AI".lower().split()
bi = BigramModel()
bi.train(corpus)

print("P('love' | 'i') =", bi.prob("love", "i"))
print("P('ai' | 'love') =", bi.prob("ai", "love"))
print("Generated:", " ".join(bi.generate(10)))


P('love' | 'i') = 0.6666666666666666
P('ai' | 'love') = 0.5
Generated: i study ai i love nlp i love ai


# Intrinsic Evaluation with Perplexity

## 1. Cross-Entropy
Given a test corpus of length $N$ and a language model $P$,  
the average negative log-likelihood is:

$$
H(P) = -\frac{1}{N}\sum_{i=1}^N \log_2 P(w_i \mid \text{context})
$$

- For a **unigram model**:
$$
P(w_i \mid \text{context}) = P(w_i)
$$

- For a **bigram model**:
$$
P(w_i \mid w_{i-1})
$$

---

## 2. Perplexity
Perplexity (PP) is the exponentiated cross-entropy:

$$
\text{PP} = 2^{H(P)} = 2^{-\frac{1}{N}\sum_{i=1}^N \log_2 P(w_i \mid \text{context})}
$$

- Intuition:  
  - Low perplexity = model assigns high probability to test data (better).  
  - High perplexity = model is "surprised" by the test data.  

---

## 3. Handling Zero Probabilities
If an unseen word/bigram occurs in test data, $P(w) = 0 \implies \log(0) = -\infty$.  
To avoid this, we use **smoothing** (e.g., Laplace / add-one smoothing):

$$
\hat{P}(w \mid h) = \frac{C(h\,w) + \alpha}{C(h) + \alpha V}
$$

where:
- $h =$ context (previous $n-1$ words)  
- $V =$ vocabulary size  
- $\alpha =$ smoothing constant (usually $1$ for add-one)  


In [5]:
import math
from collections import Counter, defaultdict

# --- Unigram Model with Laplace smoothing ---
class UnigramModel:
    def __init__(self, alpha=1.0):
        self.counts = Counter()
        self.total = 0
        self.vocab = set()
        self.alpha = alpha
    
    def train(self, corpus):
        self.counts.update(corpus)
        self.total = sum(self.counts.values())
        self.vocab = set(self.counts.keys())
    
    def prob(self, word):
        V = len(self.vocab)
        return (self.counts[word] + self.alpha) / (self.total + self.alpha * V)
    
    def perplexity(self, test_tokens):
        N = len(test_tokens)
        log_prob_sum = 0.0
        for w in test_tokens:
            p = self.prob(w)
            log_prob_sum += math.log2(p)
        H = - log_prob_sum / N
        return 2 ** H

# --- Bigram Model with Laplace smoothing ---
class BigramModel:
    def __init__(self, alpha=1.0):
        self.bigram_counts = defaultdict(Counter)
        self.context_counts = Counter()
        self.vocab = set()
        self.alpha = alpha
    
    def train(self, corpus):
        tokens = ["<s>"] + corpus + ["</s>"]
        self.vocab.update(tokens)
        for w1, w2 in zip(tokens[:-1], tokens[1:]):
            self.bigram_counts[w1][w2] += 1
            self.context_counts[w1] += 1
    
    def prob(self, w2, w1):
        V = len(self.vocab)
        return (self.bigram_counts[w1][w2] + self.alpha) / (self.context_counts[w1] + self.alpha * V)
    
    def perplexity(self, test_tokens):
        tokens = ["<s>"] + test_tokens + ["</s>"]
        N = len(tokens) - 1
        log_prob_sum = 0.0
        for w1, w2 in zip(tokens[:-1], tokens[1:]):
            p = self.prob(w2, w1)
            log_prob_sum += math.log2(p)
        H = - log_prob_sum / N
        return 2 ** H

# --- Example Usage ---
train_corpus = "I love NLP I love AI I study AI".lower().split()
test_corpus  = "I love AI".lower().split()

# Train models
uni = UnigramModel(alpha=1.0)
uni.train(train_corpus)

bi = BigramModel(alpha=1.0)
bi.train(train_corpus)

# Compute perplexity
print("Unigram perplexity on test:", uni.perplexity(test_corpus))
print("Bigram perplexity on test:", bi.perplexity(test_corpus))


Unigram perplexity on test: 4.239948049941661
Bigram perplexity on test: 4.053600464421104


# Smoothing, Interpolation, and Backoff in N-gram Models

## 1. Why Smoothing?
- Problem: Maximum Likelihood Estimation (MLE) assigns zero probability to unseen n-grams.
- Consequence: Zero probability makes perplexity $\infty$.
- Solution: **Smoothing** redistributes probability mass from seen to unseen events.

---

## 2. Additive (Laplace) Smoothing
For an $n$-gram history $h$ and next word $w$:
$$
P_{\text{Laplace}}(w \mid h) = \frac{C(h\,w) + \alpha}{C(h) + \alpha V}
$$
- $C(h\,w)$ = count of $h$ followed by $w$  
- $C(h)$ = count of context $h$  
- $V$ = vocabulary size  
- $\alpha > 0$ (usually $\alpha=1$)

---

## 3. Interpolation
- Combine higher-order and lower-order n-gram models with weights.
- Example: Trigram interpolation
$$
P(w_i \mid w_{i-2},w_{i-1}) = 
\lambda_3 \, P_{\text{tri}}(w_i \mid w_{i-2},w_{i-1})
+ \lambda_2 \, P_{\text{bi}}(w_i \mid w_{i-1})
+ \lambda_1 \, P_{\text{uni}}(w_i)
$$
- $\lambda_1+\lambda_2+\lambda_3=1$  
- $\lambda$ values tuned by cross-validation.

---

## 4. Backoff
- If higher-order $n$-gram has zero count, "back off" to lower-order model.
- Example (Katz backoff idea):
$$
P(w_i \mid w_{i-1}) = 
\begin{cases}
\frac{C(w_{i-1}w_i) - d}{C(w_{i-1})} & \text{if } C(w_{i-1}w_i) > 0 \\
\alpha(w_{i-1}) \, P(w_i) & \text{otherwise}
\end{cases}
$$
- $d$ = discount constant  
- $\alpha(w_{i-1})$ ensures probabilities normalize.

---

## 5. Summary
- **Smoothing**: prevent zero probability (Laplace, Good-Turing, Kneser–Ney).  
- **Interpolation**: mix different n-gram orders (weighted sum).  
- **Backoff**: fallback to lower-order if higher-order unseen.  

These techniques make n-gram models robust on unseen test data.


In [6]:
import random
from collections import defaultdict, Counter

# --- Base models ---
class UnigramModel:
    def __init__(self, alpha=1.0):
        self.counts = Counter()
        self.total = 0
        self.vocab = set()
        self.alpha = alpha
    
    def train(self, tokens):
        self.counts.update(tokens)
        self.total = sum(self.counts.values())
        self.vocab = set(tokens)
    
    def prob(self, w):
        V = len(self.vocab)
        return (self.counts[w] + self.alpha) / (self.total + self.alpha * V)

class BigramModel:
    def __init__(self, alpha=1.0):
        self.bigram_counts = defaultdict(Counter)
        self.context_counts = Counter()
        self.vocab = set()
        self.alpha = alpha
    
    def train(self, tokens):
        tokens = ["<s>"] + tokens + ["</s>"]
        self.vocab.update(tokens)
        for w1, w2 in zip(tokens[:-1], tokens[1:]):
            self.bigram_counts[w1][w2] += 1
            self.context_counts[w1] += 1
    
    def prob(self, w2, w1):
        V = len(self.vocab)
        return (self.bigram_counts[w1][w2] + self.alpha) / (self.context_counts[w1] + self.alpha * V)

# --- Interpolation ---
class InterpolatedModel:
    def __init__(self, uni_model, bi_model, lambdas=(0.3, 0.7)):
        assert abs(sum(lambdas) - 1.0) < 1e-6
        self.uni = uni_model
        self.bi = bi_model
        self.l1, self.l2 = lambdas
    
    def prob(self, w2, w1):
        p_uni = self.uni.prob(w2)
        p_bi = self.bi.prob(w2, w1)
        return self.l1 * p_uni + self.l2 * p_bi

# --- Simple Backoff ---
class BackoffModel:
    def __init__(self, uni_model, bi_model):
        self.uni = uni_model
        self.bi = bi_model
    
    def prob(self, w2, w1):
        # back off to unigram if bigram unseen
        if self.bi.bigram_counts[w1][w2] > 0:
            return self.bi.prob(w2, w1)
        else:
            return self.uni.prob(w2)

# --- Example ---
train_text = "I love NLP I love AI I study AI".lower().split()

uni = UnigramModel(alpha=1.0)
uni.train(train_text)

bi = BigramModel(alpha=1.0)
bi.train(train_text)

interp = InterpolatedModel(uni, bi, lambdas=(0.4, 0.6))
backoff = BackoffModel(uni, bi)

print("Smoothing (Laplace): P('ai'| 'love') =", bi.prob("ai", "love"))
print("Interpolation: P('ai'| 'love') =", interp.prob("ai", "love"))
print("Backoff: P('nlp'| 'study') =", backoff.prob("nlp", "study"))
print("Backoff (unigram fallback): P('love'| 'study') =", backoff.prob("love", "study"))


Smoothing (Laplace): P('ai'| 'love') = 0.2222222222222222
Interpolation: P('ai'| 'love') = 0.21904761904761905
Backoff: P('nlp'| 'study') = 0.14285714285714285
Backoff (unigram fallback): P('love'| 'study') = 0.21428571428571427
